# 02 — Feature Engineering

Create time-based features from the date column and visualize the engineered features.

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected+png"

from src.data_processing.load_data import load_train_data
from src.data_processing.clean_data import clean_data
from src.feature_engineering.create_time_features import add_time_features
from src.utils.helpers import load_config

config = load_config()
df = load_train_data(config)
df = clean_data(df)
df = add_time_features(df)
df.head()

[load_data] Loaded train data: 913,000 rows, 4 columns
[clean_data] Cleaned data: 913,000 rows
[feature_engineering] Added time features: year, month, quarter, day_of_week, day_of_month, week_of_year, is_weekend


,date,store,item,sales,year,month,quarter,day_of_week,day_of_month,week_of_year,is_weekend
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0
1,2013-01-01,7,12,26,2013,1,1,1,1,1,0
2,2013-01-01,7,46,27,2013,1,1,1,1,1,0
3,2013-01-01,8,12,54,2013,1,1,1,1,1,0
4,2013-01-01,9,12,35,2013,1,1,1,1,1,0


## New Columns

In [2]:
print("Columns:", list(df.columns))
df[["date","year","month","quarter","day_of_week","is_weekend","sales"]].sample(10)

Columns: ['date', 'store', 'item', 'sales', 'year', 'month', 'quarter', 'day_of_week', 'day_of_month', 'week_of_year', 'is_weekend']


,date,year,month,quarter,day_of_week,is_weekend,sales
220523,2014-03-18,2014,3,1,1,0,40
880510,2017-10-28,2017,10,4,5,1,49
449919,2015-06-19,2015,6,2,4,0,53
266909,2014-06-18,2014,6,2,2,0,12
393491,2015-02-26,2015,2,1,3,0,21
794155,2017-05-08,2017,5,2,0,0,38
123079,2013-09-04,2013,9,3,2,0,49
677082,2016-09-16,2016,9,3,4,0,26
529115,2015-11-25,2015,11,4,2,0,46
415138,2015-04-11,2015,4,2,5,1,79


## Sales by Year

In [3]:
yearly = df.groupby("year")["sales"].sum().reset_index()
fig = px.bar(yearly, x="year", y="sales",
             title="Total Sales by Year", template="plotly_dark")
fig.show()

## Sales by Quarter

In [4]:
quarterly = df.groupby(["year", "quarter"])["sales"].sum().reset_index()
quarterly["yq"] = quarterly["year"].astype(str) + "-Q" + quarterly["quarter"].astype(str)

fig = px.bar(quarterly, x="yq", y="sales",
             title="Quarterly Sales", template="plotly_dark")
fig.show()

## Weekday vs Weekend

In [5]:
weekday_avg = df.groupby("day_of_week")["sales"].mean().reset_index()
day_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
weekday_avg["name"] = weekday_avg["day_of_week"].map(lambda d: day_names[d])

fig = px.bar(weekday_avg, x="name", y="sales",
             title="Average Sales by Day of Week", template="plotly_dark")
fig.show()

In [6]:
weekend_avg = df.groupby("is_weekend")["sales"].mean().reset_index()
weekend_avg["label"] = weekend_avg["is_weekend"].map({0: "Weekday", 1: "Weekend"})

fig = px.bar(weekend_avg, x="label", y="sales",
             title="Avg Sales: Weekday vs Weekend", template="plotly_dark")
fig.show()

## Heatmap — Month vs Day of Week

In [7]:
pivot = df.pivot_table(index="day_of_week", columns="month",
                       values="sales", aggfunc="mean")

fig = px.imshow(pivot,
                labels=dict(x="Month", y="Day of Week", color="Avg Sales"),
                x=["Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec"],
                y=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"],
                color_continuous_scale="Viridis",
                title="Average Sales Heatmap: Day of Week × Month",
                template="plotly_dark")
fig.show()

## Lag Features

Lag features capture historical demand patterns by shifting sales values backward in time. They allow the model to learn from what happened 7, 14, or 30 days ago — directly encoding the auto-regressive structure inherent in time-series data. These are among the most powerful predictors in demand forecasting.

In [10]:
df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)

df["lag_7"]  = df.groupby(["store", "item"])["sales"].shift(7)
df["lag_14"] = df.groupby(["store", "item"])["sales"].shift(14)
df["lag_30"] = df.groupby(["store", "item"])["sales"].shift(30)

print("Lag feature sample (store=1, item=1):")
df[df["store"].eq(1) & df["item"].eq(1)][["date", "sales", "lag_7", "lag_14", "lag_30"]].head(35).tail(10)

Lag feature sample (store=1, item=1):


,date,sales,lag_7,lag_14,lag_30
25,2013-01-26,12,18.0,7.0,NaN
26,2013-01-27,12,15.0,10.0,NaN
27,2013-01-28,11,8.0,12.0,NaN
28,2013-01-29,6,7.0,5.0,NaN
29,2013-01-30,9,9.0,7.0,NaN
30,2013-01-31,13,8.0,16.0,13.0
31,2013-02-01,11,14.0,7.0,11.0
32,2013-02-02,21,12.0,18.0,14.0
33,2013-02-03,15,12.0,15.0,13.0
34,2013-02-04,14,11.0,8.0,10.0


## Rolling Window Features

Rolling statistics smooth out day-to-day noise and reveal the underlying demand trend. A 7-day rolling mean captures weekly patterns, while a 30-day window shows the broader monthly trend. Rolling standard deviation measures recent volatility — high values signal erratic demand.

In [11]:
df["rolling_mean_7"]  = df.groupby(["store", "item"])["sales"].transform(lambda x: x.rolling(7).mean())
df["rolling_mean_30"] = df.groupby(["store", "item"])["sales"].transform(lambda x: x.rolling(30).mean())
df["rolling_std_7"]   = df.groupby(["store", "item"])["sales"].transform(lambda x: x.rolling(7).std())

sample = df[df["store"].eq(1) & df["item"].eq(1)].copy()
fig = go.Figure()
fig.add_trace(go.Scatter(x=sample["date"], y=sample["sales"],
                         name="Raw Sales", opacity=0.4, line=dict(width=1)))
fig.add_trace(go.Scatter(x=sample["date"], y=sample["rolling_mean_7"],
                         name="7-Day Rolling Mean", line=dict(width=2)))
fig.add_trace(go.Scatter(x=sample["date"], y=sample["rolling_mean_30"],
                         name="30-Day Rolling Mean", line=dict(width=2, dash="dash")))
fig.update_layout(title="Store 1 / Item 1 — Raw Sales vs Rolling Averages",
                  template="plotly_dark",
                  xaxis_title="Date", yaxis_title="Sales")
fig.show()

## Time Index Feature

A numerical time index (`days since start`) gives models a simple linear proxy for long-term trend. Tree-based models and regression models can use this to capture gradual growth or decline in demand over the dataset's time span.

In [12]:
df["time_index"] = (df["date"] - df["date"].min()).dt.days

print(f"time_index range: {df['time_index'].min()} → {df['time_index'].max()}")
df[["date", "time_index"]].drop_duplicates().head(10)

time_index range: 0 → 1825


,date,time_index
0,2013-01-01,0
1,2013-01-02,1
2,2013-01-03,2
3,2013-01-04,3
4,2013-01-05,4
5,2013-01-06,5
6,2013-01-07,6
7,2013-01-08,7
8,2013-01-09,8
9,2013-01-10,9


## Cyclical Time Encoding

Months 1 and 12 are numerically far apart but temporally adjacent. Sine/cosine encoding maps cyclic features onto a circle so that January and December are neighbors. This prevents models from incorrectly treating the January→December jump as a large discontinuity.

In [13]:
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

cycle = df[["month", "month_sin", "month_cos"]].drop_duplicates().sort_values("month")
fig = go.Figure()
fig.add_trace(go.Scatter(x=cycle["month"], y=cycle["month_sin"], mode="lines+markers", name="sin"))
fig.add_trace(go.Scatter(x=cycle["month"], y=cycle["month_cos"], mode="lines+markers", name="cos"))
fig.update_layout(title="Cyclical Month Encoding (sin / cos)",
                  template="plotly_dark",
                  xaxis_title="Month", yaxis_title="Encoded Value",
                  xaxis=dict(tickmode="linear", dtick=1))
fig.show()

## Store & Item Aggregate Features

Global averages per store and per item encode the inherent "baseline" demand level of each entity. A high-volume store will have a higher `store_avg_sales`, giving the model a useful prior even when recent lag/rolling values are noisy.

In [14]:
df["store_avg_sales"] = df.groupby("store")["sales"].transform("mean")
df["item_avg_sales"]  = df.groupby("item")["sales"].transform("mean")

store_avg = df[["store", "store_avg_sales"]].drop_duplicates().sort_values("store")
item_avg  = df[["item", "item_avg_sales"]].drop_duplicates().sort_values("item")

fig = px.bar(store_avg, x=store_avg["store"].astype(str), y="store_avg_sales",
             title="Average Daily Sales per Store",
             template="plotly_dark", labels={"store_avg_sales": "Avg Sales", "store": "Store"})
fig.update_traces(marker_color="#22c55e")
fig.show()

In [15]:
fig = px.bar(item_avg, x=item_avg["item"].astype(str), y="item_avg_sales",
             title="Average Daily Sales per Item",
             template="plotly_dark", labels={"item_avg_sales": "Avg Sales", "item": "Item"})
fig.update_traces(marker_color="#a855f7")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Feature Visualizations

Additional charts to validate and understand the engineered features.

In [16]:
monthly_trend = df.groupby(df["date"].dt.to_period("M").astype(str))["sales"].sum().reset_index()
monthly_trend.columns = ["month", "total_sales"]

fig = px.line(monthly_trend, x="month", y="total_sales",
              title="Monthly Sales Trend (Aggregated)",
              template="plotly_dark")
fig.update_traces(line_color="#3b82f6")
fig.show()

In [17]:
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_sales = df.groupby("day_of_week")["sales"].mean().reset_index()
weekday_sales["day_name"] = weekday_sales["day_of_week"].map(lambda d: day_names[d])

fig = px.bar(weekday_sales, x="day_name", y="sales",
             title="Average Sales by Day of Week",
             template="plotly_dark", labels={"sales": "Avg Sales"})
fig.update_traces(marker_color="#f59e0b")
fig.show()

In [18]:
store_monthly = df.copy()
store_monthly["ym"] = store_monthly["date"].dt.to_period("M").astype(str)
store_perf = store_monthly.groupby(["ym", "store"])["sales"].sum().reset_index()

fig = px.line(store_perf, x="ym", y="sales", color=store_perf["store"].astype(str),
              title="Store Performance Comparison (Monthly)",
              template="plotly_dark", labels={"sales": "Total Sales", "store": "Store"})
fig.show()

## Handle Missing Values from Feature Creation

Lag and rolling window features introduce `NaN` values at the start of each store-item group (the first 7/14/30 rows have no history to look back on). We drop these rows now so that downstream models receive a clean, complete feature matrix.

In [19]:
rows_before = len(df)
null_counts = df.isna().sum()
print("NaN counts per column:")
print(null_counts[null_counts > 0])

df = df.dropna()
print(f"\nRows before: {rows_before:,}  →  Rows after: {len(df):,}  (dropped {rows_before - len(df):,})")

NaN counts per column:
lag_7               3500
lag_14              7000
lag_30             15000
rolling_mean_7      3000
rolling_mean_30    14500
rolling_std_7       3000
dtype: int64

Rows before: 913,000  →  Rows after: 898,000  (dropped 15,000)


## Save Processed Dataset

Export the fully-engineered DataFrame so that downstream notebooks and the forecasting pipeline can load it directly without rerunning all feature steps.

In [20]:
out_path = "../data/processed/processed_sales_data.csv"
df.to_csv(out_path, index=False)

print(f"Saved {len(df):,} rows × {len(df.columns)} columns → {out_path}")
print(f"\nFinal columns:\n{list(df.columns)}")

Saved 898,000 rows × 22 columns → ../data/processed/processed_sales_data.csv

Final columns:
['date', 'store', 'item', 'sales', 'year', 'month', 'quarter', 'day_of_week', 'day_of_month', 'week_of_year', 'is_weekend', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_30', 'rolling_std_7', 'time_index', 'month_sin', 'month_cos', 'store_avg_sales', 'item_avg_sales']
